## Environment setup


In [ ]:
import sys, subprocess

def _run(*args):
    subprocess.check_call([sys.executable, "-m", *args])

for pkg in ["torchaudio", "torchvision", "torch"]:
    try:
        _run("pip", "uninstall", "-y", pkg)
    except Exception:
        pass

_run("pip", "install", "torch==2.4.1", "torchvision==0.19.1",
     "--index-url", "https://download.pytorch.org/whl/cu118")

_run("pip", "install", "torch-scatter", "torch-sparse", "torch-cluster", "torch-spline-conv",
     "-f", "https://data.pyg.org/whl/torch-2.4.1+cu118.html")
_run("pip", "install", "torch-geometric")
_run("pip", "install", "fair-esm", "biotite==0.41.2", "MDAnalysis")

print("INSTALL COMPLETE — RESTART THE KERNEL")


## Sanity check

Check device type and libraries loaded.

In [ ]:
import torch
print(f"torch:          {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
assert "2.4" in torch.__version__, f"Expected torch 2.4.x, got {torch.__version__}"

import torch_geometric, torch_scatter, torch_sparse
from torch_geometric.nn import MessagePassing
print(f"torch_geometric: {torch_geometric.__version__}")
print("All imports OK")


: 

## Load model and configure paths

Load ESM-IF1, then connect paths for trajectory/topology files, cached backbone arrays, and the embedding outputs. 

In [ ]:
import numpy as np
import pandas as pd
import time
from pathlib import Path

import biotite.structure as _bs
if not hasattr(_bs, "filter_backbone") and hasattr(_bs, "filter_peptide_backbone"):
    _bs.filter_backbone = _bs.filter_peptide_backbone

import esm
import gcsfs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model, alphabet = esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
model = model.eval().to(device)
print("ESM-IF1 loaded")

METADATA_CSV = "/home/jupyter/cs229-md-prediction/data/processed/data_processed_v3_base_dataset_deduped.csv"
TRAJ_DIR     = "/home/jupyter/gcs_mount/data/trajectories/"
TOP_DIR      = "/home/jupyter/gcs_mount/data/topologies/"
BACKBONE_DIR = "gs://cs229-central/data/backbone_coords"
OUTPUT_DIR   = "gs://cs229-central/data/esmif_embeddings"

gfs = gcsfs.GCSFileSystem()

def is_gcs(p):
    return str(p).startswith("gs://")

def gcs_strip(p):
    return str(p).replace("gs://", "", 1)

def ensure_dir(p):
    if not is_gcs(p):
        Path(p).mkdir(parents=True, exist_ok=True)

ensure_dir(BACKBONE_DIR)
ensure_dir(OUTPUT_DIR)

df = pd.read_csv(METADATA_CSV)
print(f"Metadata: {len(df)} trajectories")
df.head()


## Extract backbone coordinates

For each trajectory, select the N/Cα/C backbone atoms with MDAnalysis and stack them into a `(frames, residues, 3, 3)` float32 array, which gets written to GCS. 

In [ ]:
import MDAnalysis as mda
from concurrent.futures import ProcessPoolExecutor, as_completed


def _extract_one_backbone(args):
    import MDAnalysis as mda
    import numpy as np, gcsfs, io, time
    from pathlib import Path
    idx, total, out_name, traj_path, top_path, output_file = args

    _is_gcs = output_file.startswith("gs://")
    if _is_gcs:
        _gfs = gcsfs.GCSFileSystem()
        if _gfs.exists(output_file):
            return ("skip", out_name, "")
    elif Path(output_file).exists():
        return ("skip", out_name, "")

    if not Path(traj_path).exists():
        return ("missing", out_name, "MISSING traj")
    if not Path(top_path).exists():
        return ("missing", out_name, "MISSING top")

    try:
        t0 = time.time()
        u = mda.Universe(top_path, traj_path)

        bb_N  = u.select_atoms("protein and name N")
        bb_CA = u.select_atoms("protein and name CA")
        bb_C  = u.select_atoms("protein and name C")

        n_res = len(bb_CA)
        if len(bb_N) != n_res or len(bb_C) != n_res:
            residues  = u.select_atoms("protein").residues
            n_res     = len(residues)
            n_idx     = np.full(n_res, -1, dtype=int)
            ca_idx    = np.full(n_res, -1, dtype=int)
            c_idx     = np.full(n_res, -1, dtype=int)
            all_atoms = u.atoms
            for j, res in enumerate(residues):
                for a in res.atoms:
                    if   a.name == "N":  n_idx[j]  = a.index
                    elif a.name == "CA": ca_idx[j] = a.index
                    elif a.name == "C":  c_idx[j]  = a.index
            valid            = (n_idx >= 0) & (ca_idx >= 0) & (c_idx >= 0)
            use_index_arrays = True
        else:
            use_index_arrays = False

        n_frames = len(u.trajectory)
        coords   = np.zeros((n_frames, n_res, 3, 3), dtype=np.float32)

        for i, ts in enumerate(u.trajectory):
            if use_index_arrays:
                all_pos            = all_atoms.positions
                coords[i, valid,  0] = all_pos[n_idx[valid]]
                coords[i, valid,  1] = all_pos[ca_idx[valid]]
                coords[i, valid,  2] = all_pos[c_idx[valid]]
                coords[i, ~valid]    = np.nan
            else:
                coords[i, :, 0] = bb_N.positions
                coords[i, :, 1] = bb_CA.positions
                coords[i, :, 2] = bb_C.positions

        if _is_gcs:
            buf = io.BytesIO()
            np.save(buf, coords)
            buf.seek(0)
            with _gfs.open(output_file, "wb") as f:
                f.write(buf.read())
        else:
            np.save(output_file, coords)

        return ("success", out_name, f"{coords.shape} in {time.time()-t0:.1f}s")

    except Exception as e:
        return ("fail", out_name, str(e))


def extract_backbone_coords(metadata_df, traj_dir, top_dir, output_dir, n_workers=4):
    traj_dir   = Path(traj_dir)
    top_dir    = Path(top_dir)
    out_is_gcs = str(output_dir).startswith("gs://")
    if not out_is_gcs:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    tasks = []
    for idx, row in metadata_df.iterrows():
        receptor    = row["receptor"]
        out_name    = f"{receptor.replace('~', '_')}_rep{int(row['rep'])}_{int(row['simID'])}"
        output_file = (f"{output_dir}/{out_name}.npy" if out_is_gcs
                       else str(output_dir / f"{out_name}.npy"))
        tasks.append((idx, len(metadata_df), out_name,
                      str(traj_dir / row["traj_file"]),
                      str(top_dir  / row["top_file"]),
                      output_file))

    print(f"Extracting backbone coords for {len(tasks)} trajectories ({n_workers} workers)")
    counts = {"success": 0, "skip": 0, "fail": 0, "missing": 0}
    t0 = time.time()

    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_extract_one_backbone, t): t for t in tasks}
        for fut in as_completed(futures):
            status, name, msg = fut.result()
            counts[status] = counts.get(status, 0) + 1
            if status == "success":
                print(f"  {name}: {msg}")
            elif status != "skip":
                print(f"  {name}: {status} — {msg}")

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed/60:.1f} min: {counts['success']} success, "
          f"{counts['skip']} skipped, {counts['missing']} missing, {counts['fail']} failed")


extract_backbone_coords(df, TRAJ_DIR, TOP_DIR, BACKBONE_DIR, n_workers=4)


## Encoder and GCS utilities

encode_batch feeds a batch of backbone coordinate arrays through ESM-IF1's GVP encoder and returns the per-residue hidden states. 

In [ ]:
BATCH_SIZE = 64


@torch.no_grad()
def encode_batch(model, backbone_batch, device):
    coords   = torch.from_numpy(backbone_batch).float().to(device)
    B, L     = coords.shape[0], coords.shape[1]
    nan_mask = torch.any(torch.any(~torch.isfinite(coords), dim=-1), dim=-1)
    coords   = torch.nan_to_num(coords)
    confidence = torch.ones(B, L, dtype=torch.float32, device=device)
    confidence[nan_mask] = -1.0
    encoder_out = model.encoder(coords, nan_mask, confidence)
    emb = encoder_out["encoder_out"][0].permute(1, 0, 2)
    return emb.cpu().numpy()


def _gcs_read_bytes(gcs_path, filesystem=None, max_retries=5):
    import time as _t
    _fs = filesystem or gcsfs.GCSFileSystem()
    try:
        _fs.invalidate_cache(gcs_path)
    except Exception:
        pass
    expected_size = None
    try:
        expected_size = _fs.info(gcs_path).get("size", None)
    except FileNotFoundError:
        pass
    for attempt in range(max_retries):
        try:
            raw = _fs.cat(gcs_path)
        except FileNotFoundError:
            if attempt < max_retries - 1:
                _fs = gcsfs.GCSFileSystem()
                _t.sleep(2 ** attempt)
                continue
            raise
        if expected_size is not None and len(raw) == expected_size:
            return raw
        if expected_size is None and len(raw) > 128:
            return raw
        backoff = 2 ** attempt
        print(f"  GCS partial read ({len(raw)}/{expected_size} bytes), retry {attempt+1}/{max_retries}...")
        _t.sleep(backoff)
    raise IOError(f"GCS read failed after {max_retries} retries for {gcs_path}")


def npy_load(path, _gfs_override=None):
    if is_gcs(path):
        import io
        return np.load(io.BytesIO(_gcs_read_bytes(path, filesystem=_gfs_override)))
    return np.load(path)


def npy_save(path, arr):
    if is_gcs(path):
        import io
        buf = io.BytesIO()
        np.save(buf, arr)
        buf.seek(0)
        with gfs.open(path, "wb") as f:
            f.write(buf.read())
    else:
        np.save(path, arr)


def npy_exists(path):
    return gfs.exists(path) if is_gcs(path) else Path(path).exists()


def list_npy(directory):
    if is_gcs(directory):
        prefix = gcs_strip(directory)
        return sorted(f"gs://{f}" for f in gfs.ls(prefix) if f.endswith(".npy"))
    return sorted(str(p) for p in Path(directory).glob("*.npy"))


test_files = list_npy(BACKBONE_DIR)
if test_files:
    test_emb = encode_batch(model, npy_load(test_files[0])[:2], device)
    print(f"Encoder OK: {test_emb.shape}")
else:
    print("No backbone files found")


## Embed trajectories

Main embedding loop. For each trajectory we load the first 50% of frames, run them through the encoder in fixed-size batches, and save the resulting per-residue embeddings back to GCS. Careful handling of OOM errors.

In [ ]:
FRAC = 0.5

torch.cuda.empty_cache()
backbone_files = list_npy(BACKBONE_DIR)
print(f"Found {len(backbone_files)} backbone files (frac={FRAC}, batch={BATCH_SIZE})")

for fi, fpath in enumerate(backbone_files):
    fname    = fpath.split("/")[-1] if is_gcs(fpath) else Path(fpath).name
    fstem    = fname.rsplit(".", 1)[0]
    out_name = "esmif_emb-" + fname
    out_path = (f"{OUTPUT_DIR}/{out_name}" if is_gcs(OUTPUT_DIR)
                else str(Path(OUTPUT_DIR) / out_name))

    if npy_exists(out_path):
        print(f"[{fi+1}/{len(backbone_files)}] Skip: {fstem}")
        continue

    print(f"[{fi+1}/{len(backbone_files)}] {fstem}")
    bb_full  = npy_load(fpath)
    n_use    = max(1, int(bb_full.shape[0] * FRAC))
    bb_data  = bb_full[:n_use]
    n_frames = bb_data.shape[0]
    bs       = BATCH_SIZE

    success = False
    while bs >= 1 and not success:
        print(f"    {n_frames}/{bb_full.shape[0]} frames (batch={bs})")
        embs_list = []
        t0 = time.time()
        try:
            for start in range(0, n_frames, bs):
                end = min(start + bs, n_frames)
                embs_list.append(encode_batch(model, bb_data[start:end], device))
                if end % 200 < bs or end == n_frames:
                    elapsed = time.time() - t0
                    rate    = end / max(elapsed, 1e-6)
                    eta     = (n_frames - end) / max(rate, 1e-6)
                    print(f"    Frame {end}/{n_frames}  ({rate:.1f} f/s, ETA {eta/60:.1f} min)")
            success = True
        except torch.cuda.OutOfMemoryError:
            del embs_list
            torch.cuda.empty_cache()
            bs = max(1, bs - 5)
            print(f"    OOM — retrying with batch={bs}")

    if not success:
        print(f"    FAILED: OOM at batch=1, skipping {fstem}")
        continue

    all_embs = np.concatenate(embs_list, axis=0)
    npy_save(out_path, all_embs)
    elapsed  = time.time() - t0
    print(f"  Saved {all_embs.shape} to {out_name} ({elapsed/60:.1f} min)")

    del bb_full, bb_data, embs_list, all_embs
    torch.cuda.empty_cache()

print("\nAll trajectories embedded.")


: 

## Bootstrap chunk sampling (random)

Splits trajectories into train/test by receptor, then randomly samples fixed-length temporal chunks (149 frames) from each trajectory's embedding file. 

In [ ]:
CHUNK_LEN       = 149
MIN_GAP         = CHUNK_LEN // 5
N_CHUNKS_MULTI  = 100
N_CHUNKS_SINGLE = 30
BOOTSTRAP_SEED  = 0

TRAIN_RECEPTORS_CSV = "/home/jupyter/cs229-md-prediction/data/processed/train_receptors_v3_deduped.csv"
TEST_RECEPTORS_CSV  = "/home/jupyter/cs229-md-prediction/data/processed/test_receptors_v3_deduped.csv"
EMB_BOOTSTRAP_DIR   = "gs://cs229-central/data/emb_bootstrap"

rng = np.random.default_rng(BOOTSTRAP_SEED)

train_receptors = set(pd.read_csv(TRAIN_RECEPTORS_CSV)["receptor"].unique())
test_receptors  = set(pd.read_csv(TEST_RECEPTORS_CSV)["receptor"].unique())
print(f"Train receptors: {len(train_receptors)}, Test receptors: {len(test_receptors)}")

esmif_files = {f.split("/")[-1].replace("esmif_emb-", "").replace(".npy", ""): f
               for f in list_npy(OUTPUT_DIR)}

traj_info = {}
for _, row in df.iterrows():
    receptor   = row["receptor"]
    stem       = f"{receptor.replace('~', '_')}_rep{int(row['rep'])}_{int(row['simID'])}"
    esmif_path = esmif_files.get(stem)
    if esmif_path is None:
        continue
    if   receptor in train_receptors: split = "train"
    elif receptor in test_receptors:  split = "test"
    else:                              split = "unknown"
    traj_info[stem] = {
        "receptor":   receptor,
        "rep":        int(row["rep"]),
        "simID":      int(row["simID"]),
        "label":      int(row["y"]),
        "split":      split,
        "esmif_path": esmif_path,
    }

n_train = sum(1 for v in traj_info.values() if v["split"] == "train")
n_test  = sum(1 for v in traj_info.values() if v["split"] == "test")
n_unk   = sum(1 for v in traj_info.values() if v["split"] == "unknown")
print(f"Trajectories: {len(traj_info)} (train={n_train}, test={n_test}, unmatched={n_unk})")

chunk_rows   = []
skipped      = 0

for stem, info in sorted(traj_info.items()):
    if info["split"] == "unknown":
        skipped += 1
        continue

    label    = info["label"]
    n_chunks = N_CHUNKS_MULTI if label == 1 else N_CHUNKS_SINGLE

    n_frames_total = int(df.loc[
        (df["receptor"] == info["receptor"]) &
        (df["rep"]      == info["rep"])      &
        (df["simID"]    == info["simID"]),
        "n_frames_total"
    ].iloc[0])
    n_early = max(1, int(n_frames_total * FRAC))

    if n_early < CHUNK_LEN:
        skipped += 1
        continue

    start_idxs   = []
    attempts     = 0
    max_attempts = 10 * n_chunks
    while len(start_idxs) < n_chunks and attempts < max_attempts:
        s = int(rng.integers(0, n_early - CHUNK_LEN + 1))
        if all(abs(s - prev) >= MIN_GAP for prev in start_idxs):
            start_idxs.append(s)
        attempts += 1

    for local_id, s in enumerate(start_idxs, start=1):
        chunk_rows.append({
            "traj_id":        stem,
            "receptor":       info["receptor"],
            "rep":            info["rep"],
            "simID":          info["simID"],
            "y":              label,
            "split":          info["split"],
            "chunk_id":       local_id,
            "chunk_start":    s,
            "chunk_length":   CHUNK_LEN,
            "esmif_emb_file": info["esmif_path"],
        })

emb_bootstrap = pd.DataFrame(chunk_rows)
train_chunks  = emb_bootstrap[emb_bootstrap["split"] == "train"]
test_chunks   = emb_bootstrap[emb_bootstrap["split"] == "test"]

print(f"\nTotal chunks: {len(emb_bootstrap)} ({skipped} skipped)")
print(f"  TRAIN: {len(train_chunks)}  multi={int(train_chunks['y'].sum())}  "
      f"single={int((train_chunks['y']==0).sum())}  pos_frac={train_chunks['y'].mean():.3f}")
print(f"  TEST:  {len(test_chunks)}   multi={int(test_chunks['y'].sum())}  "
      f"single={int((test_chunks['y']==0).sum())}  pos_frac={test_chunks['y'].mean():.3f}")

for name, sub in [("train", train_chunks), ("test", test_chunks), ("all", emb_bootstrap)]:
    gcs_path = f"{EMB_BOOTSTRAP_DIR}/{name}_chunks.csv"
    sub.to_csv(gcs_path, index=False, storage_options={"token": "google_default"})
    sub.to_csv(f"emb_bootstrap_{name}_chunks.csv", index=False)
    print(f"  Saved {name}: {gcs_path}")

emb_bootstrap.head(10)


## Bootstrap chunk sampling (evenly spaced)

Same chunk index construction as above, but instead of random starts the chunks are placed at uniform stride across each trajectory's time axis.

In [ ]:
CHUNK_LEN_ES       = 149
N_CHUNKS_MULTI_ES  = 100
N_CHUNKS_SINGLE_ES = 30
GCS_DIR_ES         = "gs://cs229-central/data/emb_bootstrap_evenly_spaced"

chunk_rows_es = []
skipped_es    = 0

for stem, info in sorted(traj_info.items()):
    if info["split"] == "unknown":
        skipped_es += 1
        continue

    label    = info["label"]
    n_target = N_CHUNKS_MULTI_ES if label == 1 else N_CHUNKS_SINGLE_ES

    n_frames_total = int(df.loc[
        (df["receptor"] == info["receptor"]) &
        (df["rep"]      == info["rep"])      &
        (df["simID"]    == info["simID"]),
        "n_frames_total"
    ].iloc[0])
    n_early = max(1, int(n_frames_total * FRAC))

    if n_early < CHUNK_LEN_ES:
        skipped_es += 1
        continue

    max_starts = n_early - CHUNK_LEN_ES + 1
    n_chunks   = min(n_target, max_starts)

    if n_chunks == 1:
        start_idxs = [0]
    else:
        stride     = max(1, (n_early - CHUNK_LEN_ES) // (n_chunks - 1))
        start_idxs = [i * stride for i in range(n_chunks)]

    for local_id, s in enumerate(start_idxs, start=1):
        chunk_rows_es.append({
            "traj_id":        stem,
            "receptor":       info["receptor"],
            "rep":            info["rep"],
            "simID":          info["simID"],
            "y":              label,
            "split":          info["split"],
            "chunk_id":       local_id,
            "chunk_start":    s,
            "chunk_length":   CHUNK_LEN_ES,
            "esmif_emb_file": info["esmif_path"],
        })

emb_bootstrap_es = pd.DataFrame(chunk_rows_es)
train_chunks_es  = emb_bootstrap_es[emb_bootstrap_es["split"] == "train"]
test_chunks_es   = emb_bootstrap_es[emb_bootstrap_es["split"] == "test"]

print(f"Total chunks (evenly-spaced): {len(emb_bootstrap_es)} ({skipped_es} skipped)")
print(f"  TRAIN: {len(train_chunks_es)}  multi={int(train_chunks_es['y'].sum())}  "
      f"single={int((train_chunks_es['y']==0).sum())}  pos_frac={train_chunks_es['y'].mean():.3f}")
print(f"  TEST:  {len(test_chunks_es)}   multi={int(test_chunks_es['y'].sum())}  "
      f"single={int((test_chunks_es['y']==0).sum())}  pos_frac={test_chunks_es['y'].mean():.3f}")

for name, sub in [("train", train_chunks_es), ("test", test_chunks_es), ("all", emb_bootstrap_es)]:
    gcs_path = f"{GCS_DIR_ES}/{name}_chunks.csv"
    sub.to_csv(gcs_path, index=False, storage_options={"token": "google_default"})
    sub.to_csv(f"emb_bootstrap_es_{name}_chunks.csv", index=False)
    print(f"  Saved {name}: {gcs_path}")

emb_bootstrap_es.head(10)
